## Changed File Structure
When training I used a different file structure but now I simplified it. That's why you may need to run this before project imports to run the code.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
IMPLEMENTATION_DIR = REPO_ROOT / "tiny_stories_100m"

if str(IMPLEMENTATION_DIR) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_DIR))

## NECESSARY IMPORTS

In [ ]:
from pathlib import Path

import torch
from tokenizers import Tokenizer

from tiny_stories_100m.gptModel import(
    GPTModel,
    TINYSTORIES_CONFIG_29M,
    generate,
    text_to_token_ids,
    token_ids_to_text,
)

## SET UP PROJECT PATHS AND SEE MODEL EXISTS

In [3]:
PROJECT_ROOT = Path.cwd()

RUN_NAME = "tinystories_100m" # The latest most powerful run

TOKENIZER_PATH = PROJECT_ROOT / "tinystories_tokenizer.json"
CHECKPOINT_PATH = (
    PROJECT_ROOT 
    / "checkpoints"
    / RUN_NAME
    / "best.pt"
)

assert TOKENIZER_PATH.exists(), TOKENIZER_PATH
assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device: ", device)
print("Checkpoint: ", CHECKPOINT_PATH)

Device:  cuda
Checkpoint:  C:\Users\owner\Desktop\Transformer\checkpoints\tinystories_100m\best.pt


## LOAD THE TOKENIZER AND CHECKPOINT

In [6]:
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

eos_id = tokenizer.token_to_id("<|endoftext|>")
assert eos_id is not None

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

MODEL_CONFIG = dict(TINYSTORIES_CONFIG_29M)

# assert tokenizer and model's vocab size is same
assert tokenizer.get_vocab_size() == MODEL_CONFIG["vocab_size"]

model = GPTModel(MODEL_CONFIG)

load_result = model.load_state_dict(
    checkpoint["model"],
    strict=True,
)

model = model.to(device) # loading the model to device(cuda)
model.eval() # We are not training, so we don't need to track gradients

print("Weights loaded: ", load_result)
print("Best checkpoint step: ", checkpoint["optimizer_step"])
print("Tokens seen at best checkpoint: ", f"{checkpoint["tokens_seen"]:,}")
print("Recorded best validation loss: ", checkpoint["best_val_loss"]) 



Weights loaded:  <All keys matched successfully>
Best checkpoint step:  12207
Tokens seen at best checkpoint:  99,999,744
Recorded best validation loss:  1.564269917011261


## GENERATE STORY

In [36]:
torch.manual_seed(42)

if device.type == "cuda":
    torch.cuda.manual_seed_all(42)

prompt = "Tom loves to play " 
input_ids = text_to_token_ids(
    prompt,
    tokenizer,
).to(device)

output_ids = generate(
    model=model,
    idx=input_ids,
    max_new_tokens=200,
    context_size=MODEL_CONFIG["context_length"],
    temperature=0,
    top_k=None,
    eos_id=eos_id
)

generated_text = token_ids_to_text(
    output_ids,
    tokenizer,
    skip_special_tokens=True,
)

print(generated_text)

Tom loves to play ouch on the floor. He likes to make noises and pretend he is a lion. He has a big box of toys in his room. He can make noises and pretend he is a lion.

One day, Tom's mom says, "Tom, it's time to clean up your toys. You need to put them away." Tom does not want to clean up. He wants to play more. He says, "No, mom, I don't want to clean up. I want to play more."

His mom says, "Tom, you have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys. You have to clean up your toys


## GENERATING SEVERAL SAMPLES

In [37]:
prompts = [
    "Once upon a time, there was a little dragon",
    "Lily was afraid of the dark, but one night",
    "Tom found a strange box under his bed",
    "Mia wanted to help the lost bird",
    "The little robot had never seen the rain",
]

samples = {}

for prompt_index, prompt in enumerate(prompts):
    samples[prompt] = []

    for sample_index in range(3):
        seed = 42 + prompt_index * 10 + sample_index

        torch.manual_seed(seed)
        if device.type == "cuda":
            torch.cuda.manual_seed_all(seed)

        input_ids = text_to_token_ids(
            prompt,
            tokenizer,
        ).to(device)

        output_ids = generate(
            model=model,
            idx=input_ids,
            max_new_tokens=180,
            context_size=MODEL_CONFIG["context_length"],
            temperature=0.8,
            top_k=40,
            eos_id=eos_id,
        )

        text = token_ids_to_text(
            output_ids,
            tokenizer,
            skip_special_tokens=True,
        )

        samples[prompt].append(text)

        print("=" * 90)
        print(f"Prompt: {prompt}")
        print(f"Seed: {seed}")
        print()
        print(text)

Prompt: Once upon a time, there was a little dragon
Seed: 42

Once upon a time, there was a little dragon named Sam. Sam loved to fly high in the sky. One day, while flying, he saw a big, scary monster in the sky. Sam was so scared that he ran away. He was so scared that he wanted to escape from the scary monster.

The monster tried to find a way out, but he could not. He was too small and his legs were too noisy. Sam was very sad, but he knew that he couldn't escape from the monster monster. He wished he could fly with his friends again.

But no matter how hard Sam tried, he was still stuck in the storm. He was very sad and scared too. The monster didn't know what to do and he had to find a safe place to call home.
Prompt: Once upon a time, there was a little dragon
Seed: 43

Once upon a time, there was a little dragon. He was so happy and he lived in a big dragon. One day, he decided to go on an adventure. He looked around and he saw a big lake. He asked his friend, "Can I go to the 